In [1]:
# import general libraries
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import pandas as pd
import pathlib as Path

# import ML libraries
import sklearn
import tensorflow as tf
import keras
import numba
import torch

import jax


# other ML imports
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score, cohen_kappa_score
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


from torch import optim
from torch import nn
from tqdm import tqdm

import torchvision

import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader


import torchmetrics
import copy


# define data directory
DATA_DIR = Path.Path("Data")

In [2]:
with np.load(DATA_DIR / "train.npz") as data:
    print("Datasets:", data.files, "\n")
    
    X = data["X"]
    Y = data["y"]

    if X.shape[0] == Y.shape[0]:
        print("Shape of Training Data:", X.shape)
        
    classNames = data["classes"]
    print("Class Names:", classNames)
    classes = np.unique(Y)
    print("Classes:", classes)
    print("Number of classes:", len(classes), "\n")


    for lenC in range(len(classes)):
        print(f"Size of class {lenC} - {classNames[lenC]}:", len(Y[np.isin(Y, classes[lenC])]))
    print("\n")

    bands = data["bands"]
    print("Number of bands:", len(bands))
    print("Bands:", bands, "\n")

    wavelength_nm = data["wavelength_nm"]
    print("Number of wavelengths:", len(wavelength_nm))
    print("Wavelength (nm):", wavelength_nm, "\n")

    resolution_m = data["resolution_m"]
    print("Number of resolutions:", len(resolution_m))
    print("Resolution (m):", resolution_m, "\n")

samplesTrain = (X.shape[0])
H = (X.shape[1])
W = (X.shape[2])

Datasets: ['X', 'y', 'classes', 'bands', 'wavelength_nm', 'resolution_m'] 

Shape of Training Data: (18900, 64, 64, 13)
Class Names: ['AnnualCrop' 'Forest' 'HerbaceousVegetation' 'Highway' 'Industrial'
 'Pasture' 'PermanentCrop' 'Residential' 'River' 'SeaLake']
Classes: [0 1 2 3 4 5 6 7 8 9]
Number of classes: 10 

Size of class 0 - AnnualCrop: 2100
Size of class 1 - Forest: 2100
Size of class 2 - HerbaceousVegetation: 2100
Size of class 3 - Highway: 1750
Size of class 4 - Industrial: 1750
Size of class 5 - Pasture: 1400
Size of class 6 - PermanentCrop: 1750
Size of class 7 - Residential: 2100
Size of class 8 - River: 1750
Size of class 9 - SeaLake: 2100


Number of bands: 13
Bands: ['B01' 'B02' 'B03' 'B04' 'B05' 'B06' 'B07' 'B08' 'B8A' 'B09' 'B10' 'B11'
 'B12'] 

Number of wavelengths: 13
Wavelength (nm): [ 443  490  560  665  705  740  783  842  865  945 1375 1610 2190] 

Number of resolutions: 13
Resolution (m): [60 10 10 10 20 20 20 10 20 60 60 20 20] 



In [3]:
appleMetal = (torch.backends.mps.is_available())  
nvidiaCUDA = (torch.cuda.is_available())

if appleMetal == True and nvidiaCUDA == False:
    print("Using Apple Silicon Metal GPU Backend")
    backendDevice = "mps"

elif appleMetal == False and nvidiaCUDA == True:
    print("Using NVIDIA CUDA GPU Backend")
    backendDevice = "cuda"

elif appleMetal == False and nvidiaCUDA == False:
    print("Guess we slowmaxxing and using a CPU today")
    backendDevice = "cpu"

elif appleMetal == True and nvidiaCUDA == True:
    print("I would have asked python to print MONEY but I guess imma ask you")

Using Apple Silicon Metal GPU Backend


In [4]:
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.30, random_state=69, stratify=Y)

mean = X_train.mean(axis=(0, 1, 2))
stdev = X_train.std(axis=(0, 1, 2))

mu = torch.as_tensor(mean, dtype=torch.float32, device=backendDevice).view(1, 13, 1, 1)
sd = torch.as_tensor(stdev, dtype=torch.float32, device=backendDevice).view(1, 13, 1, 1)

In [5]:
trainDataset = TensorDataset(torch.from_numpy(X_train), torch.as_tensor(Y_train, dtype=torch.long))
valDataset = TensorDataset(torch.from_numpy(X_val), torch.as_tensor(Y_val, dtype=torch.long))

trainLoader = DataLoader(trainDataset, batch_size=32, shuffle=True)
valLoader = DataLoader(valDataset, batch_size=32, shuffle=False)

In [6]:
def normalizeImages(images):
    # NumPy layout: (batch 0, height 1, width 2, bands 3)
    # PyTorch layout: (batch 0, bands 3, height 1, width 2) use permute
    # Turn to float, reshape to PyTorch shape, send to device, and normalize using mean and std
    images = images.float().permute(0, 3, 1, 2).contiguous() # 0 3 1 2
    images = images.to(backendDevice)
    imgNorm = (images - mu) / (sd + 1e-6) # no div by 0 ffs
    return imgNorm

In [7]:
# input image size = 64 x 64
# in_channels = 13
# num_classes = 10

# btw increasing stride is essetially like downsampling, 1 moves i pixed, 2 moves 2 pixels, and so on. capture all spatial vs downsample -- genralization effects quantify 
# padding acts as a buffer arounf the image data, essentailly 0s to let kernels go out of bounds a little bit

class RegularizedCNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(RegularizedCNN, self).__init__()

        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.avgpool = nn.AvgPool2d(kernel_size=2, stride=2)

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(8) 
        
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)

        self.conv3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(32)

        self.conv4 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)

        self.dropout = nn.Dropout(p=0.5) 
        
        self.fc1 = nn.Linear(64 * 8 * 8, num_classes)

    def forward(self, x):
        # Conv -> Batch Norm -> Leaky ReLU -> Pool
        # Conv -> Batch Norm -> Leaky ReLU

        x = self.maxpool(F.leaky_relu(self.bn1(self.conv1(x))))

        x = self.maxpool(F.leaky_relu(self.bn2(self.conv2(x))))

        x = self.maxpool(F.leaky_relu(self.bn3(self.conv3(x))))

        x = (F.leaky_relu(self.bn4(self.conv4(x))))


        x = x.reshape(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc1(x)

        return x


In [8]:
def trainModel(model, trainLoader, valLoader, lossFx, optimizerFx, num_epochs, seed, device):

    best_val_f1 = float('-inf')
    best_model_state = None

    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, targets in tqdm(trainLoader, desc="Training"):
            images = normalizeImages(images).to(device) 
            targets = targets.to(device)

            optimizerFx.zero_grad(set_to_none=True) # Optimization
            outputs = model(images)
            loss = lossFx(outputs, targets)
            loss.backward()
            optimizerFx.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / len(trainLoader)

        model.eval()
        val_predictions = []
        val_targets = []

        with torch.inference_mode():
            for images, targets in valLoader:
                images = normalizeImages(images).to(device)
                targets = targets.to(device)

                outputs = model(images)
                
                val_predictions.append(outputs.argmax(dim=1).cpu().numpy())
                val_targets.append(targets.cpu().numpy())

        val_predictions = np.concatenate(val_predictions)
        val_targets = np.concatenate(val_targets)

        val_accuracy = accuracy_score(val_targets, val_predictions)
        val_f1 = f1_score(val_targets, val_predictions, average='macro', zero_division=0)

        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, "
              f"Val Accuracy: {val_accuracy:.4f}, Val F1 (Macro): {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = copy.deepcopy(model.state_dict())
            print(f"Best model saved! Epoch {epoch+1}, F1: {best_val_f1:.4f}")


    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nTraining complete. Loaded best model with Val F1: {best_val_f1:.4f}")

    return model



def saveModel(model, file_path):
    torch.save(model.state_dict(), file_path)
    print(f"Model saved to {file_path}")
    return 0

In [9]:
numEpochs = 40
modelSeeds = [0, 2, 4, 6, 9, 17, 42, 67, 69, 420, 4200, 6969, 69420]



for i, seed in enumerate(modelSeeds):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    print(f"\nTraining model with seed {seed}...")
    model = RegularizedCNN(in_channels=13, num_classes=10).to(backendDevice)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    print(model)

    trained_model = trainModel(model, trainLoader, valLoader, loss_function, optimizer, numEpochs, seed, backendDevice)

    model_file_path = DATA_DIR / f"leaky_RegCNN_{i}.pth"
    saveModel(trained_model, model_file_path)


Training model with seed 0...
RegularizedCNN(
  (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (avgpool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv1): Conv2d(13, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=4096, out_features=10, bias=Tr

Training: 100%|██████████| 414/414 [00:03<00:00, 126.06it/s]


Epoch [1/40], Train Loss: 0.7921, Val Accuracy: 0.8148, Val F1 (Macro): 0.8059
Best model saved! Epoch 1, F1: 0.8059


Training: 100%|██████████| 414/414 [00:03<00:00, 130.83it/s]


Epoch [2/40], Train Loss: 0.4792, Val Accuracy: 0.8619, Val F1 (Macro): 0.8566
Best model saved! Epoch 2, F1: 0.8566


Training: 100%|██████████| 414/414 [00:02<00:00, 145.40it/s]


... (1,031 outputs left out)

Training: 100%|██████████| 414/414 [00:02<00:00, 154.06it/s]


Epoch [39/40], Train Loss: 0.0641, Val Accuracy: 0.9499, Val F1 (Macro): 0.9482


Training: 100%|██████████| 414/414 [00:02<00:00, 154.20it/s]


Epoch [40/40], Train Loss: 0.0621, Val Accuracy: 0.9545, Val F1 (Macro): 0.9529

Training complete. Loaded best model with Val F1: 0.9568
Model saved to Data/leaky_RegCNN_12.pth


In [15]:
loaded_model = RegularizedCNN(in_channels=13, num_classes=10)

loaded_model.load_state_dict(torch.load(DATA_DIR / "leaky_RegCNN_6.pth", map_location=backendDevice, weights_only=True))

loaded_model = loaded_model.to(backendDevice)
loaded_model.eval()

print(loaded_model)

RegularizedCNN(
  (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (avgpool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv1): Conv2d(13, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=4096, out_features=10, bias=True)
)


In [16]:
def plot_confusion(y_true, y_pred, title):

    fig, ax = plt.subplots(figsize=(12, 10))

    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        labels=np.arange(len(classNames)),
        display_labels=classNames,
        cmap="Blues",
        values_format="d",
        xticks_rotation=45,
        ax=ax
    )

    ax.set_title(title, fontsize=16)
    ax.tick_params(axis="both", labelsize=11)

    plt.tight_layout()
    plt.show()

In [17]:
loaded_model.eval()
val_predictions = []

with torch.inference_mode():
    for images, targets in valLoader:
        images = normalizeImages(images)

        scores = loaded_model(images)
        val_predictions.append(scores.argmax(dim=1).cpu().numpy())

# Combine all batches into one array
val_predictions = np.concatenate(val_predictions)

print(val_predictions.shape)

(5670,)


In [18]:
loaded_model.eval()
train_predictions = []
train_targets = []

with torch.inference_mode():
    for images, targets in trainLoader:
        images = normalizeImages(images)
        train_targets.append(targets.cpu().numpy())

        scores = loaded_model(images)
        train_predictions.append(scores.argmax(dim=1).cpu().numpy())

train_predictions = np.concatenate(train_predictions)
train_targets = np.concatenate(train_targets)

In [28]:
# # Validation confusion matrix
# plot_confusion(Y_val, val_predictions, "Best CNN — Validation")

# # Training confusion matrix
# plot_confusion(train_targets, train_predictions, "Best CNN — Training")


In [36]:
ensembleLogits = []

for i in range (len(modelSeeds)):
    modelName = f"leaky_RegCNN_{i}.pth"
    print(f"\nLoading model from {modelName}...")
    loaded_model = RegularizedCNN(in_channels=13, num_classes=10)
    loaded_model.load_state_dict(torch.load(DATA_DIR / modelName, map_location=backendDevice, weights_only=True))
    loaded_model = loaded_model.to(backendDevice)
    loaded_model.eval()

    print(loaded_model)

    val_predictions = []
    val_logits = []

    with torch.inference_mode():
        for images, targets in valLoader:
            images = normalizeImages(images)

            scores = loaded_model(images)
            val_predictions.append(scores.argmax(dim=1).cpu())
            val_logits.append(scores.cpu())

    val_predictions = torch.cat(val_predictions)
    val_logits = torch.cat(val_logits)

    print(val_predictions.shape)
    print(val_logits.shape)


    probabilities = torch.softmax(val_logits, dim=1)
    ensembleLogits.append(probabilities)



Loading model from leaky_RegCNN_0.pth...
RegularizedCNN(
  (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (avgpool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv1): Conv2d(13, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=4096, out_features=

In [60]:
meanLogits = torch.mean((ensembleLogits), dim=0)

print(meanLogits[0])

print(meanLogits[0].argmax())


tensor([1.9226e-04, 1.8902e-09, 7.6285e-06, 1.7358e-06, 7.5077e-09, 1.3625e-07,
        9.9980e-01, 2.2562e-06, 1.8691e-07, 7.0801e-11])
tensor(6)
